# 07 — Gold Layer: Fact Trips
Consolidates all Silver collections into a unified Fact Table for the Star Schema.

In [ ]:
from src.core.spark import get_spark
from src.core.mongo import read_mongo, write_mongo
from src.core.config import settings
import pyspark.sql.functions as F

In [ ]:
spark = get_spark("TLC_Gold_Fact")

In [ ]:
def read_and_flatten_silver(collection_name):
    df = read_mongo(spark, "tlc_silver", collection_name)
    return df.select(
        F.date_format(F.col("datetimes.pickup"), "yyyyMMdd").alias("pickup_date_id"),
        F.date_format(F.col("datetimes.dropoff"), "yyyyMMdd").alias("dropoff_date_id"),
        F.col("locations.pickup.zone_id").alias("pickup_zone_id"),
        F.col("locations.dropoff.zone_id").alias("dropoff_zone_id"),
        F.coalesce(F.col("vendor_id"), F.lit(-1)).alias("vendor_id"),
        F.coalesce(F.col("metrics.rate_code_id"), F.lit(-1)).alias("rate_code_id"),
        F.coalesce(F.col("financials.payment_type"), F.lit("Unknown")).alias("payment_type_id"),
        F.col("_meta.vehicle_type").alias("vehicle_id"),
        F.coalesce(F.col("metrics.distance_miles"), F.lit(0.0)).alias("trip_distance"),
        F.coalesce(F.col("datetimes.duration_minutes"), F.lit(0.0)).alias("duration_minutes"),
        F.coalesce(F.col("metrics.passenger_count"), F.lit(0.0)).alias("passenger_count"),
        F.coalesce(F.col("financials.fare_amount"), F.lit(0.0)).alias("fare_amount"),
        F.coalesce(F.col("financials.tip_amount"), F.lit(0.0)).alias("tip_amount"),
        F.coalesce(F.col("financials.tolls_amount"), F.lit(0.0)).alias("tolls_amount"),
        F.coalesce(F.col("financials.congestion_surcharge"), F.lit(0.0)).alias("congestion_surcharge"),
        F.coalesce(F.col("financials.cbd_congestion_fee"), F.lit(0.0)).alias("cbd_congestion_fee"),
        F.coalesce(F.col("financials.total_amount"), F.lit(0.0)).alias("total_amount")
    )

In [ ]:
collections = ["trips_yellow", "trips_green", "trips_fhv", "trips_hvfhv"]
dfs = []
for coll in collections:
    try:
        dfs.append(read_and_flatten_silver(coll))
    except Exception as e:
        print(f"Collection {coll} not found or error occurred: {e}")

In [ ]:
from functools import reduce

if dfs:
    fact_trips_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dfs)
else:
    print("No silver collections found.")

In [ ]:
if dfs:
    write_mongo(fact_trips_df, "tlc_gold", "fact_trips", mode="overwrite")
    print("Wrote fact_trips to tlc_gold.")